# SGSMA 2026 — Workshop Homework

**Anomaly Detection in Power Systems with AI** · *Masoud Barati*

Workshop materials & full notebook: <https://github.com/msdbarati>

---

## What to do (≈ 20 minutes)

1. Run every cell **top-to-bottom** (Shift + Enter).
2. Fill in the two **HOMEWORK** cells (clearly marked `### YOUR CODE`).
3. Run the last cell — it prints a **Results Card** and writes `results.csv`.
4. Send me **(a) a screenshot of the Results Card** and **(b) `results.csv`** to claim your certificate:

> **Email:** **masoud.barati@pitt.edu**   ·   **Subject:** `SGSMA 2026 Homework — <your name>`

That's it. Should run on a laptop CPU in under 2 minutes.

---

## Requirements
`numpy · scikit-learn · torch · matplotlib` — already in any standard scientific Python environment.


In [ ]:
# 0. Setup -------------------------------------------------------------
# !pip install -q numpy scikit-learn torch matplotlib
import numpy as np, torch, torch.nn as nn, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

SEED = 7
np.random.seed(SEED); torch.manual_seed(SEED)
NAME = "YOUR FULL NAME"     # <-- write your name here, it appears on the results card

print("Hello,", NAME)


## 1. Tiny synthetic PMU dataset (3 classes)

We use a smaller version of the workshop dataset:
**300 windows × 4 channels × 48 samples** across three classes:

| id | class | signature |
|----|-------|-----------|
| 0 | Normal | steady values + noise |
| 1 | Fault  | voltage dip + phase jump |
| 2 | Oscillation | low-damped sinusoid in `f` |


In [ ]:
T, C, N_PER = 48, 4, 60         # 48 samples · 4 channels · 60 per class (small on purpose!)
F0, FS = 60.0, 120.0
dt = 1.0 / FS

def normal():
    V = 1.0 + 0.04*np.random.randn(T)      # heavier noise so models aren't trivially perfect
    I = 0.8 + 0.05*np.random.randn(T)
    f = F0 + 0.10*np.random.randn(T)
    theta = np.cumsum(2*np.pi*(f-F0)*dt) + 0.08*np.random.randn(T)
    return np.stack([V, I, f, theta], axis=0)

def fault():
    x = normal(); s = np.random.randint(8, T-12); dur = np.random.randint(6, 14)
    x[0, s:s+dur] -= 0.18*np.random.uniform(0.7, 1.1)
    x[3, s:s+dur] += np.deg2rad(10)*np.random.uniform(0.6, 1.1)
    return x

def osc():
    x = normal(); t = np.arange(T)*dt
    x[2] += 0.05*np.sin(2*np.pi*np.random.uniform(0.6,1.4)*t) * np.exp(-0.4*t)
    return x

X, y = [], []
for _ in range(N_PER): X.append(normal()); y.append(0)
for _ in range(N_PER): X.append(fault());  y.append(1)
for _ in range(N_PER): X.append(osc());    y.append(2)
X = np.array(X, dtype=np.float32); y = np.array(y, dtype=np.int64)
mu, sd = X.mean((0,2), keepdims=True), X.std((0,2), keepdims=True)+1e-6
X = (X - mu) / sd

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
CLASSES = ["Normal","Fault","Oscillation"]
print("shape", X.shape, " split:", Xtr.shape, Xte.shape)


### 1.1 Visualize one window per class

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 4), sharex=True)
for i, name in enumerate(CLASSES):
    s = X[np.where(y==i)[0][0]]
    axes[i].plot(np.arange(T)*dt*1000, s[0], label="V")
    axes[i].plot(np.arange(T)*dt*1000, s[2], label="f", alpha=0.7)
    axes[i].set_ylabel(name); axes[i].grid(alpha=0.3)
axes[-1].set_xlabel("ms"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()


## 2. Baseline weak-inductive model — small CNN

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(C, 16, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(32, 3),
        )
    def forward(self, x): return self.net(x)

def train(model, epochs=20):
    Xt = torch.tensor(Xtr); yt = torch.tensor(ytr)
    Xv = torch.tensor(Xte); yv = torch.tensor(yte)
    opt = torch.optim.Adam(model.parameters(), 2e-3); cel = nn.CrossEntropyLoss()
    for _ in range(epochs):
        idx = torch.randperm(len(Xt))
        for s in range(0, len(Xt), 64):
            b = idx[s:s+64]
            loss = cel(model(Xt[b]), yt[b])
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        pred = model(Xv).argmax(1).numpy()
    return pred

pred_cnn = train(CNN())
acc_cnn  = accuracy_score(yte, pred_cnn)
f1_cnn   = f1_score(yte, pred_cnn, average="macro")
print(f"CNN baseline:   acc={acc_cnn:.3f}   F1={f1_cnn:.3f}")


## 3. Strong-inductive baseline — provided **rubric**

These are the IF-THEN rules an LLM typically emits after looking at a few labeled examples.

In [ ]:
def features(w):
    V, I, f, theta = w
    return dict(
        dV         = float(np.abs(V).max() - np.abs(V).min()),
        dtheta_deg = float(np.rad2deg(np.diff(theta)).max()),
        rocof_max  = float(np.abs(np.gradient(f, dt)).max()),
        rocof_mean = float(np.abs(np.gradient(f, dt)).mean()),
    )

def rubric(w):
    f = features(w)
    if f["dV"] > 1.5 and f["dtheta_deg"] > 20:           return 1   # Fault
    if f["rocof_max"] > 2.0 and f["rocof_mean"] > 0.8:   return 2   # Oscillation
    return 0                                                          # Normal

pred_rub = np.array([rubric(w) for w in Xte])
acc_rub  = accuracy_score(yte, pred_rub)
f1_rub   = f1_score(yte, pred_rub, average="macro")
print(f"Rubric only:    acc={acc_rub:.3f}   F1={f1_rub:.3f}")


## 4. **HOMEWORK 1** — Add one new rubric rule

Look at the misclassified test samples and add **one extra rule** of your own to `my_rubric`
to boost the score above the provided rubric. Anything goes — a new threshold, a Z-score
on `V`, a spectral check, etc.

> Replace the `### YOUR CODE HERE ###` block with your rule.


In [ ]:
def my_rubric(w):
    f = features(w)
    if f["dV"] > 1.5 and f["dtheta_deg"] > 20:           return 1   # Fault
    if f["rocof_max"] > 2.0 and f["rocof_mean"] > 0.8:   return 2   # Oscillation

    ### YOUR CODE HERE ##############################################
    # Hint: add ONE extra IF that catches the misclassified samples.
    # Example template:
    # if <some condition on f or w>:
    #     return <0, 1 or 2>
    #################################################################

    return 0

pred_my  = np.array([my_rubric(w) for w in Xte])
acc_my   = accuracy_score(yte, pred_my)
f1_my    = f1_score(yte, pred_my, average="macro")
print(f"YOUR rubric:    acc={acc_my:.3f}   F1={f1_my:.3f}")
print(f"Improvement over provided rubric: ΔAcc = {acc_my - acc_rub:+.3f}, ΔF1 = {f1_my - f1_rub:+.3f}")


## 5. **HOMEWORK 2** — Try one other deep model

Pick **one** of `LSTM`, `GRU`, or `Transformer` and train it on the same data.
You can copy the patterns below or write your own. Replace `MyModel` with your choice.

In [ ]:
# Choose ONE — uncomment / write your own
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        ### YOUR CODE HERE ############################################
        # Example LSTM template (uncomment):
        # self.lstm = nn.LSTM(C, 32, batch_first=True)
        # self.fc   = nn.Linear(32, 3)
        ###############################################################
        self.lstm = nn.LSTM(C, 32, batch_first=True)
        self.fc   = nn.Linear(32, 3)

    def forward(self, x):
        ### YOUR CODE HERE ############################################
        x = x.transpose(1, 2)
        o, _ = self.lstm(x)
        return self.fc(o[:, -1, :])
        ###############################################################

pred_my_model = train(MyModel())
acc_my_model  = accuracy_score(yte, pred_my_model)
f1_my_model   = f1_score(yte, pred_my_model, average="macro")
print(f"YOUR model:     acc={acc_my_model:.3f}   F1={f1_my_model:.3f}")


## 6. Results Card — please send me this!

Run the cell below.  It will print a small table **and** save `results.csv` next to this notebook.
**Submit both the table screenshot and the CSV** to claim your certificate.

In [ ]:
import csv, datetime as dt, json, platform

rows = [
    ("CNN baseline",        acc_cnn,      f1_cnn),
    ("Rubric (provided)",   acc_rub,      f1_rub),
    ("Rubric (YOUR rule)",  acc_my,       f1_my),
    ("YOUR model",          acc_my_model, f1_my_model),
]

print("="*60)
print(f"SGSMA 2026 Homework — Results Card")
print(f"Name      : {NAME}")
print(f"Date      : {dt.date.today().isoformat()}")
print(f"Python    : {platform.python_version()}  ·  Torch {torch.__version__}")
print("-"*60)
print(f"{'Method':25s}{'Accuracy':>12s}{'Macro-F1':>12s}")
for m, a, f in rows:
    print(f"{m:25s}{a:12.3f}{f:12.3f}")
print("-"*60)
print(f"Best method : {max(rows, key=lambda r:r[1])[0]}")
print(f"Δ vs CNN    : ΔAcc={max(r[1] for r in rows)-acc_cnn:+.3f},  ΔF1={max(r[2] for r in rows)-f1_cnn:+.3f}")
print("="*60)

with open("results.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["name", "date", "method", "accuracy", "macro_f1"])
    for m, a, fs in rows:
        w.writerow([NAME, dt.date.today().isoformat(), m, f"{a:.4f}", f"{fs:.4f}"])
print("\nSaved → results.csv  (in the same folder as this notebook)")


---

### Submission

Email me **(a)** a screenshot of the Results Card and **(b)** the `results.csv` file:

> **Subject:** `SGSMA 2026 Homework — <your name>`
> **To:** **masoud.barati@pitt.edu**

Bonus points: open a pull request on the workshop repo
<https://github.com/msdbarati> with your `results.csv` and a one-paragraph note on
what your custom rubric rule does.

Thanks for attending — and have fun! 🚀
